# Applying SAM 3 to map text PNGs

In [ ]:
# Imports
from pathlib import Path
from zipfile import ZipFile
from geopandas import read_file as read_geo_file
from PIL import Image
from torchvision.transforms import PILToTensor
from torch import cat as tensor_cat
from transformers import Sam3Processor, Sam3Model

In [ ]:
# Assessing data files inside zip file
with ZipFile(r"./Os-Historic-Batch1.zip") as zip:
    print(*(x for x in zip.namelist() if x.endswith(".gpkg")), sep = ",\n")

In [ ]:
# Check this matches one of the file names printed out above
point_prompts_fp = "pngs/text-locations.gpkg"

In [ ]:
# Load point prompts data
with ZipFile(r"./Os-Historic-Batch1.zip", "r") as zip:
    with zip.open(r"pngs/text-locations.gpkg", "r") as temp:
        point_prompts_meta = read_geo_file(temp)

point_prompts_meta.head()

## Extracting point prompts

`point_prompts` should be a nested list of dimensions $(N, M_n, 1, 2)$ where:
- $N$ equals the number of pngs
- $M_n$ equals the number of labelled text instances within image $n$.
- $1$ represents that each text instance in the GB1900 gazetteer is only labelled with a single point.
- $2$ represents the $x,y$ coordinates for prompt point.

`point_labels` is a nested list of dimensions $(N, M_n, 1)$, that indicates whether the point prompt is "positive" $(1)$ or "negative" $(0)$. **NOTE** all points are positive.

In [ ]:
image_filenames = sorted(point_prompts_meta['png_filename'].unique())
point_prompts = []
point_labels = []
for png in image_filenames:
    temp = point_prompts_meta.loc[
        (point_prompts_meta['png_filename'] == png), ["pixel_x", "pixel_y"]
    ]
    point_prompts.append(temp.values[:, None, :].tolist())
    point_labels.append([[1]] * len(temp))

## Extracting PNGs

In [ ]:
# Load pngs and convert to Tensors
transform = PILToTensor()
images = []
with ZipFile(r"./Os-Historic-Batch1.zip", "r") as zip:
    for png in image_filenames:
        with zip.open(f"pngs/{png}", "r") as temp:
            images.append(transform(Image.open(temp, "r", )))

images = tensor_cat(images, dim = 0)